# Bijection factory — extend the precomputes beyond p < 1024

**Run this BEFORE `phi_factory.ipynb`** (it supplies the bijections the Φ factory would
otherwise have to mint itself, and grows every downstream capability: CRT primes for Hilbert
polynomials, p-isogeny pairs for Φ_p).

**Just hit Run All.** Both slow steps checkpoint — the rigid l-set searches persist per disc in
`data/rigid_lset_cache.json`, the bijections per prime in `data/ecqf_ord_pcbij_ext.json`
(same format as the shipped `ecqf_ord_pcbij_4_1024.json`) — so interrupting and re-running
resumes where it left off.

The pipeline: enumerate classes (a, p), 0 < a < 2√p, p prime in (1024, N]; group by the exact
discriminant d = a²−4p; try a rigid spanning set for each d; compute + store the bijection of
every class over a spanning disc. Discs without a spanning set (or with a non-Atkin prime in
their conductor) are reported at the end together with a **vote** on which new modular
polynomials Φ_l would unblock the most of them — that's what decides the next `phi_factory` targets.

**Getting results back**: commit `data/ecqf_ord_pcbij_ext.json` and `data/rigid_lset_cache.json`.

In [ ]:
import os
os.chdir('../../pycode')
from bij_factory import *

# optional GPU trace tables (needs pytorch; falls back to numpy silently)
try:
    import trace_gpu
    print('trace backend:', trace_gpu.enable())
except ImportError:
    print('trace backend: numpy (pip install torch to enable cuda/mps)')

N = 2048              # bump this once the timing looks right (4096, 8192, ...)
by_d = aps_by_disc(1024, N)
print(f'{sum(len(v) for v in by_d.values())} classes over {len(by_d)} distinct discriminants, '
      f'discs down to {min(by_d)}')

In [ ]:
# rigid l-set search per disc (slow once, cached forever)
parts = partition_discs(list(by_d))

In [ ]:
# the main run: compute + store every computable bijection (resumable)
stats = compute_and_save(1024, N)
stats

In [ ]:
# spot-check: disc grouping of stored bijections vs the independent ancestor-data path
import random
from qfs import qf_disc
from hilbert_crt import class_endo_discs
store = load_ext_bijections()
random.seed(0)
for ap in random.sample(sorted(store), min(12, len(store))):
    groups = {}
    for j, qf in store[ap].items():
        groups.setdefault(qf_disc(qf), []).append(j)
    want = class_endo_discs(*ap)
    assert {d: sorted(js) for d, js in groups.items()} == want, ap
print('spot-check vs class_endo_discs: OK')

In [ ]:
# the vote: which new modular polynomials unblock the most remaining discs?
tally = vote_for_new_modpolys(parts)

After this finishes, `phi_factory.ipynb` will automatically reuse the stored bijections for
every prime this run covered (no recomputation), and the vote above tells you which targets to
pass to `run_factory(...)` first.

## The Hilbert factory
Turn everything harvested so far into Hilbert polynomials: every disc whose available CRT primes
(shipped + extension store + hilbert_roots_ext) clear its coefficient bound gets its certified
$H_d$ merged into `data/hilbpolys_crt.json`. Checkpointed; re-running skips finished discs.

In [ ]:
stats = hilbert_factory()
stats